# Style Plots and Animations

A `PlotStyle` sets the look of the plots and animations of the flow, Voronoi, proportional and symbol cartograms in one place: the surface and text colors, the boundary, the palettes by data type, the color scale and the colorbar. A call decides which data a plot shows, the style decides how it looks by default, and an argument of the call always wins over the style.

## Import Libraries and Compute a Cartogram

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import HTML
from matplotlib import rc

import carto_flow as cf
import carto_flow.data as examples
import carto_flow.flow_cartogram as flow
from carto_flow import Channel, Palettes, PlotStyle, get_style, register_style, unregister_style

states = examples.load_us_census(population=True)
cartogram = flow.create_flow_cartogram(
    states, "Population", options=flow.MorphOptions(snapshot_every=10, area_scale=1e-6, show_progress=False)
)

## Choose a style

Three styles are built in: `"default"` (the look of the library), `"dark"` and `"classic"` (the look before styles: matplotlib and library defaults). The default style is used when nothing else is asked for.

In [ ]:
p = cartogram.plot(facecolor="Population")
p.ax.set_title("default")
p.ax.axis("off")

`style=` styles one plot:

In [ ]:
for name in ("dark", "classic"):
    p = cartogram.plot(facecolor="Population", style=name)
    p.ax.set_title(name)
    p.ax.axis("off")

For a block of code or a whole session, use `style_context` or `set_style`:

In [ ]:
with cf.style_context("dark"):
    p = cartogram.plot(facecolor="Population")
    p.ax.set_title("in a dark block")
    p.ax.axis("off")

p = cartogram.plot(facecolor="Population")  # the block is over: the default style again
p.ax.set_title("after the block")
p.ax.axis("off")

cf.set_style("dark")  # the session, like matplotlib.pyplot.style.use
p = cartogram.plot(facecolor="Population")
p.ax.set_title("session style")
p.ax.axis("off")
cf.set_style(None)  # back to the default

The style of a call is the `style=` argument, else the block, else the session, else `"default"`.

Plots that take `style=`: `Cartogram.plot`, `plot_cartogram`, `plot_comparison`, the field and convergence plots and the `animate_*` functions of the flow cartogram; `VoronoiCartogram.plot`, `plot_displacement`, `plot_comparison`, `plot_compactness`, the topology plots, the convergence plot and `animate_voronoi_history`; `plot_partitions` and `plot_dot_density` of the proportional cartogram; `SymbolCartogram.plot`, `plot_symbols`, `plot_comparison`, `plot_adjacency`, `plot_adjacency_heatmap`, `plot_tiling` and `plot_displacement` of the symbol cartogram; the density and velocity `preview_modulator` and the tiling plots (`TilingResult.plot`, `plot_tile`, `plot_prototile`).

A figure that you create yourself with `plt.subplots` gets the surface of the style with `get_style().apply(fig)`; a plot function that is given `ax=` styles the axes and the text, not your figure.

## What the style sets for each type of data

The plot function decides the role of the data from the data itself, and the style supplies the palette of that role:

| data | role | palette |
|---|---|---|
| a numeric column or array | sequential | `palettes.sequential` (the default style truncates `YlGnBu` so that no fill is close to the surface) |
| `facecolor="errors_pct"`, or any data with `center=` | diverging, symmetric around the center | `palettes.diverging` |
| a text, categorical or boolean column, or `categorical=True` | categorical | `palettes.categorical` |
| a direction (`VelocityPlotOptions(color_by="direction")`) | cyclic | `palettes.cyclic` |

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
get_style().apply(fig)
cartogram.plot(facecolor="Region", ax=axes[0, 0])  # categories
cartogram.plot(facecolor="Population", center=1e7, ax=axes[0, 1])  # diverging around 10 million
cartogram.plot(facecolor="Population", norm="linear", ax=axes[1, 0])  # a linear scale
cartogram.plot(facecolor="Population", cmap="Reds", ax=axes[1, 1])  # a palette of the call
for ax, title in zip(
    axes.ravel(),
    ["categorical", "diverging, center=1e7", 'norm="linear"', 'cmap="Reds"'],
    strict=True,
):
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

The default style takes a logarithmic scale for positive data that spans more than one decade (`color_scale="auto"`) and writes the tick labels in real numbers (`1M`, `2M`, `5M`). `norm=` accepts a matplotlib `Normalize` or one of `"linear"`, `"sqrt"`, `"log"`, `"auto"`; the magnitudes of the velocity field (`VelocityPlotOptions(norm=...)`) and of the Voronoi displacement (`plot_displacement(norm=...)`) take the same scales.

To keep a linear scale for data that spans more than one decade, pass `norm="linear"` to the call, set `color_scale="linear"` in a style (`PlotStyle.default().copy_with(color_scale="linear")`), or raise the threshold with `log_span` (the number of decades that triggers `"auto"`, default 1). A `vmin` or `vmax` in the call also fixes a linear scale; combined with `norm="log"` they are the limits of the logarithmic scale.

## Write your own style

Start from a built-in style and change what you need:

In [ ]:
mine = PlotStyle.default().copy_with(
    palettes=Palettes(sequential="Blues", sequential_range=(0.2, 1.0), diverging="PuOr"),
    channels={"edge": Channel(value="#444444"), "linewidth": Channel(value=0.5)},
    color_scale="linear",
    rc_update={"font.size": 12},  # merges into the matplotlib rcParams of the style
)
register_style("mine", mine)

p = cartogram.plot(facecolor="Population", style="mine")
p.ax.set_title("mine")
p.ax.axis("off")

- `rc` holds matplotlib rcParams (surface, text and tick colors, font sizes, grid, legend). `copy_with(rc=...)` replaces them, `copy_with(rc_update=...)` merges.
- `PlotStyle.from_mplstyle("ggplot")` or `PlotStyle.from_mplstyle("my.mplstyle")` builds a style from a matplotlib style sheet.
- `palettes` holds the colormap of each role. `channels["edge"].value` and `channels["linewidth"].value` are the boundary color and width of the regions.
- `channels` also sets the fixed fill (`"fill"`), opacity (`"alpha"`) and the boundary of the symbols; with `range=(low, high)` a channel sets the output range of the data-driven `alpha`, `linewidth` and `label` (font size) of `plot_symbols`, and with `scale` (`"linear"`, `"sqrt"`, `"log"`, `"auto"`, `"rank"`) how the data is spread over that range; `alpha_norm=`, `linewidth_norm=` and `label_fontsize_norm=` of the call win. `"edge"` is the faint boundary that separates symbols; `"stroke"` is the color of an edge that carries data (a data-driven `linewidth` or a `hatch`). `hatch_cycle` is the sequence of hatch patterns assigned to the categories of `hatch=`.
- `reference_facecolor`, `reference_edgecolor`, `reference_linewidth` and `reference_alpha` are the layer of context geometries (the base of a dot density map, the complement of `plot_partitions`, the tiles of `plot_tiling`); `annotation_size` is the font size of the labels placed on a map (`label_fontsize` of the Voronoi plots).
- `series_color`, `accent_color`, `positive_color` and `highlight_color` are the colors of the diagnostic overlays of the Voronoi plots and animation: the convergence curve and the cells and centroids of an animation, the marks of a deviation (displacement targets, violated adjacency), of a good outcome (convergence, reassigned cells) and of a case to look at (satellite cells). The velocity arrows of the flow plots take `accent_color`. The base layers of the Voronoi plots use the reference fill and the axes edge color, which also colors the reference lines of the convergence plot.
- `color_scale` (`"linear"`, `"log"`, `"auto"`), `tick_format` (`"plain"`, `"si"`) and the `colorbar_*` fields set the colorbar.
- A field left `None` leaves matplotlib or the library default alone, so `PlotStyle()` is the classic look.
- `style.rc_params()` returns the rcParams, to use the look in your own matplotlib code with `plt.rc_context(style.rc_params())`. A figure that you created yourself gets the surface with `style.apply(fig)`.

`"default"`, `"dark"` and `"classic"` cannot be replaced; `register_style(name, style, overwrite=True)` replaces another registered name, and `unregister_style(name)` removes one.

`"default"`, `"dark"` and `"classic"` cannot be replaced; `register_style(name, style, overwrite=True)` replaces another registered name, and `unregister_style(name)` removes one.

In [ ]:
unregister_style("mine")

### Keep your styles between sessions

Registered styles live in the memory of the running Python process; nothing is saved to disk, so they are gone when the session ends. To have them in every session, register them in code that runs at startup:

- Put the `register_style(...)` calls in a module of your own, for example `my_styles.py` next to your project or on your Python path, and import it (`import my_styles`) at the top of a script or notebook.
- Or put the calls in a startup script that runs automatically: an IPython startup file (`~/.ipython/profile_default/startup/10-carto-flow-styles.py`) for interactive sessions and notebooks, or a file named by the `PYTHONSTARTUP` environment variable for the plain Python prompt.

```python
# my_styles.py
from carto_flow import PlotStyle, register_style, set_style

register_style("house", PlotStyle.default().copy_with(rc_update={"font.size": 12}))
set_style("house")      # optional: make it the style of the session
```

A matplotlib style sheet (`PlotStyle.from_mplstyle("my.mplstyle")`) can hold the `rc` settings in a file, but not the palettes, channels, color scale or colorbar fields.

## Combine styles

A field that a style leaves `None` is left alone, so a style can set only part of the look and be layered over another. `a | b` (or `a.merge(b)`) is a style in which whatever `b` sets wins: the `rc` dicts merge, the palettes and the channels merge field by field, and the other fields of `b` that are not `None` replace those of `a`.

In [ ]:
talk_fonts = PlotStyle(rc={"xtick.labelsize": 20, "ytick.labelsize": 20, "axes.labelsize": 22})
big_dark = PlotStyle.dark() | talk_fonts

`style=`, `style_context` and `set_style` also take a sequence of styles or names, merged in order:

In [ ]:
p = cartogram.plot(facecolor="Population", style=("dark", talk_fonts))
p.ax.set_title("dark with large type")
p.ax.axis("off")

## Animations

The animations take `style=` like the plots, and the frames are drawn with the style. The size of a figure is not part of a style: pass `figsize=` (and `dpi=` when saving).

In [ ]:
rc("animation", html="jshtml")

anim = flow.animation.animate_morph_history(
    cartogram,
    facecolor="errors_pct",
    legend=True,
    figsize=(8, 5),
    show_axes=False,
    duration=2.0,
    fps=10,
    style="dark",
)
plt.close()
HTML(anim.to_jshtml())

`morph_timeline` turns the snapshots of a cartogram into keyframes that hold on the first state, ease through the morph, hold on the last state and return. The animation of the keyframes takes the style of the first call, here the default style with the large type of a slide:

In [ ]:
from carto_flow.flow_cartogram.animation import animate_geometry_keyframes, linear_over, morph_timeline

keyframes, times = morph_timeline(cartogram, hold_start=1.0, morph=2.0, hold_end=1.0, back=2.0)
anim = animate_geometry_keyframes(
    keyframes,
    keyframe_times=times,
    position_mapper=linear_over("time"),
    duration=times[-1],
    fps=10,
    facecolor="Population",
    legend=True,
    figsize=(8, 5),
    style=("default", talk_fonts),
)
plt.close()
HTML(anim.to_jshtml())

`back=0` leaves out the return, and `easing="linear"` plays the morph at constant speed. To save an animation, `save_animation(anim, "map.mp4", fps=30, dpi=120)` writes the frames with the surface of the style.